In [1]:

import warnings
import torch
import numpy as np

from rich import print
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import KFold, StratifiedKFold
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabNetModelConfig, TabTransformerConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device == "cuda":
    torch.set_float32_matmul_precision('high')

# With Human Threats, With Cross Validation

In [3]:
model, data = final_extinctionrisk_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

mapping = {'Lower_risk':1, 'Higher_risk':2}
data['Extinction_risk'] = data['Extinction_risk'].map(mapping)

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

Data Shape: (9053, 32) | # of cat cols: 14 | # of num cols: 18

 Features: ['Beak_length_culmen', 'Beak_depth', 'Tarsus_length', 'Wing_length', 'Hand_wing_index', 'Tail_length', 
'Minimum_latitude', 'Maximum_latitude', 'Minimum_elevation', 'Elevational_range', 'Maximum_elevation', 
'Habitat_breadth', 'Diet_breadth', 'Adult_survival_annual', 'Generation_length', 'Range_size', 'Body_mass', 
'Clutch_size', 'Order', 'Family', 'Agriculture', 'Hunting', 'Invasive_species', 'Climate_change', 
'Primary_lifestyle', 'Island_restricted_breeding', 'Latitudinal_range', 'Realm', 'Diet', 'Habitat', 'Migration', 
'Extinction_risk']

Target: Extinction_risk

In [4]:
data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=256,
    max_epochs=40,
    early_stopping_patience=3,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

In [5]:
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

2026-04-10 15:15:45,531 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [6]:
import csv
import pandas as pd

train_indices = []
test_indices = []
for i in range(1,11):
    with open("./datasets/r_folds/fold_"+str(i)+"_train_idx.csv", newline='') as csvfile:
        train_df = pd.read_csv(csvfile)
        train_list = train_df.to_numpy().flatten().tolist()
        train_indices.append(train_list)
    with open("./datasets/r_folds/fold_"+str(i)+"_test_idx.csv", newline='') as csvfile:
        test_df = pd.read_csv(csvfile)
        test_list = test_df.to_numpy().flatten().tolist()
        test_indices.append(test_list)

In [7]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


train = data
datamodule = None
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for fold, (train_idx, val_idx) in enumerate(zip(train_indices, test_indices)):
        print("Fold: "+str(fold))
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]
        if datamodule is None:
            # Initialize datamodule and model in the first fold
            # uses train data from this fold to fit all transformers
            datamodule = tabular_model.prepare_dataloader(
                train=train_fold, validation=val_fold, seed=42
            )
            model = tabular_model.prepare_model(datamodule)
        else:
            # Creates a copy of the datamodule with same transformers but different train and validation data
            datamodule = datamodule.copy(train=train_fold, validation=val_fold)
        # Train the model
        tabular_model.train(model, datamodule)
        pred_df = tabular_model.predict(val_fold)
        

        y_t = val_fold['Extinction_risk']
        acc = accuracy_score(y_t, pred_df["Extinction_risk_prediction"].values)

        f1 = f1_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1]) #1 is lower risk, 2 is higher risk.
        precision = precision_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1])
        recall = recall_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1])
        

        acc_metrics.append(acc)
        f1_metrics.append(f1)
        precision_metrics.append(precision)
        recall_metrics.append(recall)
        

        # Reset the trained weights before next fold
        tabular_model.model.reset_weights()

Fold: 0

2026-04-10 15:15:45,594 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-10 15:15:45,609 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-10 15:15:45,666 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabNetModel
2026-04-10 15:15:45,726 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-10 15:15:45,727 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which 

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 26.3 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 26.3 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.3 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 122                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

2026-04-10 15:21:31,686 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-10 15:21:31,725 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-10 15:21:31,733 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 26.3 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 26.3 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.3 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 122                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

2026-04-10 15:22:07,532 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-04-10 15:22:07,533 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [8]:
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
print(f"KFold Mean f1: {np.mean(f1_metrics)} | KFold SD: {np.std(f1_metrics)}")
print(f"KFold Mean Precision: {np.mean(precision_metrics)} | KFold SD: {np.std(precision_metrics)}")
print(f"KFold Mean Recall: {np.mean(recall_metrics)} | KFold SD: {np.std(recall_metrics)}")

f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

KFold Mean Acc: 0.9977918846730818 | KFold SD: 0.004886549448296141

KFold Mean f1: 0.996389823306508 | KFold SD: 0.009689384656614602

KFold Mean Precision: 0.9972734564069624 | KFold SD: 0.0069734473896826054

KFold Mean Recall: 0.9955377589478556 | KFold SD: 0.01290236542988121

KFold Mean Precision: Higher Risk 0.99645, Lower Risk 0.9981  | KFold SD: (np.float64(0.0089), np.float64(0.00407))

KFold Mean Recall: Higher Risk 0.99189, Lower Risk 0.99918 | KFold SD: (np.float64(0.01738), np.float64(0.00204))

KFold Mean f1: Higher Risk 0.99414, Lower Risk 0.99864 | KFold SD: (np.float64(0.01298), np.float64(0.00301))

# Without Human Threats, With Cross Validation

In [9]:
model, data = final_extinctionrisk_noth_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

mapping = {'Lower_risk':1, 'Higher_risk':2}
data['Extinction_risk'] = data['Extinction_risk'].map(mapping)

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=256,
    max_epochs=40,
    early_stopping_patience=3,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


train = data
datamodule = None
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for fold, (train_idx, val_idx) in enumerate(zip(train_indices, test_indices)):
        print("Fold: "+str(fold))
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]
        if datamodule is None:
            # Initialize datamodule and model in the first fold
            # uses train data from this fold to fit all transformers
            datamodule = tabular_model.prepare_dataloader(
                train=train_fold, validation=val_fold, seed=42
            )
            model = tabular_model.prepare_model(datamodule)
        else:
            # Creates a copy of the datamodule with same transformers but different train and validation data
            datamodule = datamodule.copy(train=train_fold, validation=val_fold)
        # Train the model
        tabular_model.train(model, datamodule)
        pred_df = tabular_model.predict(val_fold)
        

        y_t = val_fold['Extinction_risk']
        acc = accuracy_score(y_t, pred_df["Extinction_risk_prediction"].values)

        f1 = f1_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1]) #1 is lower risk, 2 is higher risk.
        precision = precision_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1])
        recall = recall_score(y_t, pred_df["Extinction_risk_prediction"].values, average=None, labels=[2, 1])
        

        acc_metrics.append(acc)
        f1_metrics.append(f1)
        precision_metrics.append(precision)
        recall_metrics.append(recall)
        

        # Reset the trained weights before next fold
        tabular_model.model.reset_weights()

Data Shape: (9053, 28) | # of cat cols: 10 | # of num cols: 18

 Features: ['Beak_length_culmen', 'Beak_depth', 'Tarsus_length', 'Wing_length', 'Hand_wing_index', 'Tail_length', 
'Minimum_latitude', 'Maximum_latitude', 'Minimum_elevation', 'Elevational_range', 'Maximum_elevation', 
'Habitat_breadth', 'Diet_breadth', 'Adult_survival_annual', 'Generation_length', 'Range_size', 'Body_mass', 
'Clutch_size', 'Order', 'Family', 'Primary_lifestyle', 'Island_restricted_breeding', 'Latitudinal_range', 'Realm', 
'Diet', 'Habitat', 'Migration', 'Extinction_risk']

Target: Extinction_risk

2026-04-10 15:22:07,721 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


Fold: 0

...

2026-04-10 15:27:07,273 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-04-10 15:27:07,307 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-10 15:27:07,316 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 25.8 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 25.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 25.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 118                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

2026-04-10 15:27:41,088 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-04-10 15:27:41,088 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [10]:
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
print(f"KFold Mean f1: {np.mean(f1_metrics)} | KFold SD: {np.std(f1_metrics)}")
print(f"KFold Mean Precision: {np.mean(precision_metrics)} | KFold SD: {np.std(precision_metrics)}")
print(f"KFold Mean Recall: {np.mean(recall_metrics)} | KFold SD: {np.std(recall_metrics)}")

f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

KFold Mean Acc: 0.998454258290342 | KFold SD: 0.0032065198714291954

KFold Mean f1: 0.9974628725313176 | KFold SD: 0.006435706918582872

KFold Mean Precision: 0.998596305233512 | KFold SD: 0.003031697356090318

KFold Mean Recall: 0.9963903252593582 | KFold SD: 0.01129366787450817

KFold Mean Precision: Higher Risk 0.99882, Lower Risk 0.99838  | KFold SD: (np.float64(0.00237), 
np.float64(0.00356))

KFold Mean Recall: Higher Risk 0.99305, Lower Risk 0.99973 | KFold SD: (np.float64(0.01525), np.float64(0.00055))

KFold Mean f1: Higher Risk 0.99588, Lower Risk 0.99905 | KFold SD: (np.float64(0.0086), np.float64(0.00197))